### Experimental historical illustration basic categorization work

Experimental historical newspaper illustration basic categorization demo.

Acknowledgments

<table><tr><td>
<img src="https://digi.kansalliskirjasto.fi/images/sosiaali_en.png" alt="European Regional Development Fund" height="110">
</td><td>
<img src="https://digi.kansalliskirjasto.fi/images/en_EU_rgb.png" alt="Leverage from the EU 2014-2020" height="110"></td>
</tr></table>

#### Setup

**Note:** This notebook has been updated to use TensorFlow 2.x (using  for frozen  graph loading). 

Download the image classifier and the labels.

Note! the download links below will inactivate at December 2019, after which please download the packages (Illustration base type classifier model file & labels) from https://digi.kansalliskirjasto.fi/opendata and update links below.

In [ ]:


!curl https://digi.kansalliskirjasto.fi/opendata-download-file/035f2c8bfc124d159b066272913c67e8/nlf_basetype_classifier.pb --output nlf_basetype_classifier.pb
    
!curl https://digi.kansalliskirjasto.fi/opendata-download-file/976df7ccbcef4831a270e6254263cfd6/nlf_basetype_classifier_labels.txt  --output nlf_basetype_classifier_labels.txt
    
    

###  Image categorization code

Basic imports

In [ ]:
# TF2: removed %tensorflow_version 1.x magic (Colab-only, TF1 specific)
import tensorflow as tf
import sys
import glob
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Disable TF2 eager execution to allow loading a frozen .pb graph
tf.compat.v1.disable_eager_execution()

# setup for label changes
usedstrings = {
    'piirros painokuva': 'piirrospainokuva',
    'teksti mainos': 'mainos',
    'roskat': 'muut'
}


In [ ]:
# The function to do the categorization, the img parameter carries the filename

def luokittelekuva(img=sys.argv[1]):
    # TF2: use tf.io.gfile instead of tf.gfile
    image_data = tf.io.gfile.GFile(img, 'rb').read()

    AIDIR = ""  # for codecollab
    image_path = "./tmp/"

    # Read labels
    # TF2: use tf.io.gfile instead of tf.gfile
    label_lines = [line.rstrip() for line
                   in tf.io.gfile.GFile(AIDIR + "nlf_basetype_classifier_labels.txt")]

    # Load frozen graph
    # TF2: use tf.compat.v1.GraphDef and a new explicit Graph object
    graph_def = tf.compat.v1.GraphDef()
    with tf.io.gfile.GFile(AIDIR + "nlf_basetype_classifier.pb", 'rb') as f:
        graph_def.ParseFromString(f.read())

    graph = tf.compat.v1.Graph()
    with graph.as_default():
        tf.compat.v1.import_graph_def(graph_def, name='')

    # Run inference
    # TF2: use tf.compat.v1.Session with the explicit graph
    with tf.compat.v1.Session(graph=graph) as sess:
        softmax_tensor = sess.graph.get_tensor_by_name('final_result:0')
        predictions = sess.run(softmax_tensor,
                               {'DecodeJpeg/contents:0': image_data})

        # Sort to show labels of first prediction in order of confidence
        limit_score = 0.6
        top_k = predictions[0].argsort()[-len(predictions[0]):][::-1]

        max_score = 0
        max_id = 0
        for node_id in top_k:
            score = predictions[0][node_id]
            if max_score < score:
                max_score = score
                max_id = node_id

        human_string = label_lines[max_id]
        print('%s : %s (score = %.5f)' % (image_path, human_string, score))

        if human_string in usedstrings:
            human_string = usedstrings[human_string]

        return (human_string, score)


One expected drawing type:

In [ ]:

!curl https://digi.kansalliskirjasto.fi/sanomalehti/binding/1198629/articles/43569800/images/1437505?scale=1.0 --output kuva1.jpg


In [ ]:
# The actual way to run the function

luo, score = luokittelekuva("kuva1.jpg") # sys.argv[1])


In [ ]:
print("Category: {0}, score {1} ".format(luo, score)) 

Illustration of unknown type:

In [ ]:
!curl https://digi.kansalliskirjasto.fi/aikakausi/binding/1355524/articles/78396775/images/120919660?scale=1.0 --output kuva2.jpg

In [ ]:
luo, score = luokittelekuva("kuva2.jpg") # sys.argv[1])
print("Category: {0}, score {1} ".format(luo, score))


In [ ]:

!curl https://digi.kansalliskirjasto.fi/sanomalehti/binding/679866/articles/1544609/images/10576494?scale=1.0 --output kuva3.jpg

In [ ]:
# then not so succesfull classification...

luo, score = luokittelekuva("kuva3.jpg") 
print("Category: {0}, score {1} ".format(luo, score))
